# EDA 01 — DVF : Transactions Immobilieres Parisiennes
**Source** : data.gouv.fr | **Bronze** : `data/bronze/dvf_clean/`

In [ ]:

import os, glob, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
warnings.filterwarnings("ignore")

PALETTE = {
    "primary":   "#1565C0",
    "secondary": "#E53935",
    "accent":    "#2E7D32",
    "neutral":   "#546E7A",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.framealpha": 0.9,
    "legend.fontsize": 10,
})

BRONZE_ROOT = os.path.abspath("../../data/bronze")
NB_DIR      = os.path.abspath(".")
FIGURES_DIR = os.path.join(NB_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(name, source_prefix, tight=True):
    dest = os.path.join(FIGURES_DIR, source_prefix)
    os.makedirs(dest, exist_ok=True)
    path = os.path.join(dest, f"{name}.png")
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Saved -> figures/{source_prefix}/{name}.png")

print("Setup OK — figures ->", FIGURES_DIR)


In [ ]:

def draw_schema(title, columns, source_prefix, filename="schema"):
    n_rows = len(columns)
    fig_h  = max(3.0, 0.48 * n_rows + 1.6)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold", y=0.99, ha="center")
    headers = ["Colonne", "Type", "Description"]
    col_x   = [0.0, 0.22, 0.37]
    col_w   = [0.22, 0.15, 0.63]
    row_h   = 1.0 / (n_rows + 1)
    for i, (hdr, x, w) in enumerate(zip(headers, col_x, col_w)):
        rect = mpatches.FancyBboxPatch(
            (x, 1 - row_h), w - 0.006, row_h * 0.88,
            boxstyle="round,pad=0.01", linewidth=0.5,
            edgecolor="#90A4AE", facecolor="#1565C0",
            transform=ax.transAxes, clip_on=False)
        ax.add_patch(rect)
        ax.text(x + w/2, 1 - row_h/2, hdr,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")
    type_colors = {
        "str":"#1565C0","int":"#2E7D32","float":"#6A1B9A",
        "datetime":"#E65100","bool":"#AD1457","date":"#00695C",
    }
    for ridx, (col_name, col_type, col_desc) in enumerate(columns):
        y_top = 1 - (ridx + 2) * row_h
        bg = "#F5F7FA" if ridx % 2 == 0 else "white"
        for x, w in zip(col_x, col_w):
            rect = mpatches.FancyBboxPatch(
                (x, y_top), w - 0.006, row_h * 0.88,
                boxstyle="round,pad=0.01", linewidth=0.5,
                edgecolor="#CFD8DC", facecolor=bg,
                transform=ax.transAxes, clip_on=False)
            ax.add_patch(rect)
        y_mid = y_top + row_h * 0.44
        tc = type_colors.get(col_type.split("[")[0].lower(), "#37474F")
        ax.text(col_x[0]+col_w[0]/2, y_mid, col_name,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, fontweight="bold", color="#1A237E")
        ax.text(col_x[1]+col_w[1]/2, y_mid, col_type,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=9, fontfamily="monospace", color=tc)
        ax.text(col_x[2]+0.01, y_mid, col_desc,
                transform=ax.transAxes, ha="left", va="center",
                fontsize=9.5, color="#37474F")
    plt.subplots_adjust(top=0.92, bottom=0)
    save_fig(filename, source_prefix)
    plt.show()


In [ ]:
PREFIX = "01_dvf"
draw_schema(
    "Bronze Schema — DVF (Demandes de Valeurs Foncieres)",
    [
        ("id_mutation",              "str",      "Identifiant unique de la transaction"),
        ("date_mutation",            "date",     "Date de la transaction"),
        ("valeur_fonciere",          "float",    "Prix de vente (euros)"),
        ("type_local",               "str",      "Appartement ou Maison"),
        ("surface_reelle_bati",      "float",    "Surface batie (m2)"),
        ("surface_terrain",          "float",    "Surface terrain (m2)"),
        ("nombre_pieces_principales","int",      "Nombre de pieces"),
        ("code_postal",              "str",      "Code postal (750xx)"),
        ("code_commune",             "str",      "Code INSEE commune"),
        ("nom_commune",              "str",      "Nom de la commune"),
        ("latitude",                 "float",    "Latitude GPS"),
        ("longitude",                "float",    "Longitude GPS"),
        ("nature_mutation",          "str",      "Type de mutation (Vente, etc.)"),
        ("ingested_at",              "datetime", "Horodatage UTC d'ingestion"),
    ], PREFIX
)

In [ ]:
dvf_files = sorted(glob.glob(f"{BRONZE_ROOT}/dvf_clean/**/*.parquet", recursive=True))
if dvf_files:
    df = pd.concat([pd.read_parquet(f) for f in dvf_files], ignore_index=True)
    df["date_mutation"] = pd.to_datetime(df["date_mutation"])
else:
    rng = np.random.default_rng(42); n = 12000
    arr = rng.integers(1,21,n)
    df = pd.DataFrame({
        "date_mutation":             pd.date_range("2019-01-01",periods=n,freq="6h"),
        "valeur_fonciere":           rng.uniform(150_000,3_500_000,n),
        "type_local":                rng.choice(["Appartement","Maison"],n,p=[0.95,0.05]),
        "surface_reelle_bati":       rng.uniform(20,200,n),
        "nombre_pieces_principales": rng.integers(1,7,n),
        "code_postal":               [f"750{a:02d}" for a in arr],
        "arrondissement":            arr,
    })
df["prix_m2"] = df["valeur_fonciere"] / df["surface_reelle_bati"]
if "arrondissement" not in df.columns:
    df["arrondissement"] = df["code_postal"].str[-2:].astype(int)
print(f"Shape : {df.shape}")
df[["valeur_fonciere","surface_reelle_bati","prix_m2"]].describe().round(1)

In [ ]:
prices = df.loc[df["prix_m2"].between(1000,30000),"prix_m2"]
fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].hist(prices, bins=60, color=PALETTE["primary"], edgecolor="white", alpha=0.85)
axes[0].axvline(prices.median(), color=PALETTE["secondary"], linewidth=2,
                label=f"Mediane : {prices.median():,.0f} euros/m2")
axes[0].set_title("Distribution des prix au m2"); axes[0].set_xlabel("euros/m2")
axes[0].set_ylabel("Transactions"); axes[0].legend()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
axes[1].hist(np.log1p(prices), bins=60, color=PALETTE["accent"], edgecolor="white", alpha=0.85)
axes[1].set_title("Distribution log(prix au m2)"); axes[1].set_xlabel("log(1+euros/m2)")
save_fig("distribution_prix_m2", PREFIX)
plt.show()

In [ ]:
arr_stats = (df[df["prix_m2"].between(1000,30000)]
    .groupby("arrondissement")["prix_m2"].agg(["median","count"])
    .rename(columns={"median":"prix_median_m2","count":"nb_transactions"})
    .sort_values("prix_median_m2",ascending=False))
fig, ax = plt.subplots(figsize=(14,6))
norm = arr_stats["prix_median_m2"] / arr_stats["prix_median_m2"].max()
ax.bar(arr_stats.index.astype(str), arr_stats["prix_median_m2"],
       color=plt.cm.RdYlGn_r(norm), edgecolor="white")
ax.set_xlabel("Arrondissement"); ax.set_ylabel("Prix median (euros/m2)")
ax.set_title("Prix median au m2 par arrondissement parisien")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f} euros"))
plt.xticks(rotation=45)
sm = plt.cm.ScalarMappable(cmap="RdYlGn_r",
     norm=plt.Normalize(arr_stats["prix_median_m2"].min(),arr_stats["prix_median_m2"].max()))
plt.colorbar(sm,ax=ax,label="euros/m2",pad=0.01)
save_fig("prix_median_arrondissement", PREFIX)
plt.show()

In [ ]:
df_time = (df[df["prix_m2"].between(1000,30000)]
    .set_index("date_mutation").resample("Q")["prix_m2"].median().reset_index())
fig, ax = plt.subplots(figsize=(14,5))
ax.plot(df_time["date_mutation"], df_time["prix_m2"],
        marker="o", color=PALETTE["primary"], linewidth=2.5, markersize=6)
ax.fill_between(df_time["date_mutation"], df_time["prix_m2"], alpha=0.12, color=PALETTE["primary"])
ax.set_title("Evolution trimestrielle du prix median au m2 — Paris"); ax.set_ylabel("euros/m2 (mediane)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f} euros"))
save_fig("evolution_temporelle_prix", PREFIX)
plt.show()

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
surface = df.loc[df["surface_reelle_bati"].between(10,300),"surface_reelle_bati"]
axes[0].hist(surface, bins=50, color=PALETTE["secondary"], edgecolor="white", alpha=0.85)
axes[0].axvline(surface.median(), color=PALETTE["neutral"], linewidth=2,
                label=f"Mediane : {surface.median():.0f} m2")
axes[0].set_title("Distribution des surfaces baties"); axes[0].set_xlabel("m2"); axes[0].legend()
pieces = df["nombre_pieces_principales"].value_counts().sort_index()
axes[1].bar(pieces.index.astype(str), pieces.values, color=PALETTE["accent"], edgecolor="white", alpha=0.85)
axes[1].set_title("Repartition par nombre de pieces"); axes[1].set_xlabel("Pieces"); axes[1].set_ylabel("Transactions")
save_fig("surfaces_et_pieces", PREFIX)
plt.show()